# 实验报告03：正则表达式与网络结构化数据

**学号：** 20244081123　　**姓名：** 钟涛

实验内容：正则表达式的基础匹配、分组/回溯引用/替换、使用 BeautifulSoup 解析 HTML。

In [1]:
import hashlib
import os
import platform
import uuid
import sys

# 修改为自己的学号
STUDENT_ID = 20244081123

CONDA_ENV = os.environ.get("CONDA_DEFAULT_ENV")
print(f"[Conda 环境] {CONDA_ENV}")

MACHINE_FP = hashlib.sha256(
    f"{platform.node()}|{platform.machine()}|{platform.system()}|{uuid.getnode()}".encode()
).hexdigest()[:16]

RUN_ID = f"{STUDENT_ID}-{MACHINE_FP}"
print(f"[学号]       {STUDENT_ID}")
print(f"[机器指纹] {MACHINE_FP}")
print(f"[RUN_ID]   {RUN_ID}")
print(f"[Python]    {sys.version.split()[0]}")

[Conda 环境] exp01
[学号]       20244081123
[机器指纹] b47d9754c73871cb
[RUN_ID]   20244081123-b47d9754c73871cb
[Python]    3.10.22


## 实验1：正则表达式的基础匹配

先导入 `re`。核心函数：`re.findall(pattern, string)` 以列表形式返回所有不重叠的匹配。

In [2]:
import re

text = "cat on mat; later: cats and dogs"

print(r"数字 \d+      :", re.findall(r"\d+", text))
print(r"空白 \s+      :", re.findall(r"\s+", text))
print(r"整词 \bcat\b :", re.findall(r"\bcat\b", text))
print(r"对照: 普通 cat:", re.findall(r"cat", text))

数字 \d+      : []
空白 \s+      : [' ', ' ', ' ', ' ', ' ', ' ']
整词 \bcat\b : ['cat']
对照: 普通 cat: ['cat', 'cat']


**匹配规则说明：**

- `\d+`：一个或多个数字；目标文本中没有数字，因此结果为空列表。
- `\s+`：一个或多个空白字符（空格、制表符、换行等），匹配到 6 段空格。
- `\bcat\b`：`\b` 是单词边界，只匹配独立的单词 `cat`，所以不会匹配 `cats` 中的 `cat`。
- 对照的 `cat`（没有边界约束）会同时匹配 `cat` 和 `cats` 的前三个字母，说明 `\b` 的作用是限定整词。

In [3]:
text2 = "cat dog bird CAT"
print("cat|dog          :", re.findall(r"cat|dog", text2))
print("a|aa|aaa -> aaaa :", re.findall(r"a|aa|aaa", "aaaa"))
print("aaa|aa|a -> aaaa :", re.findall(r"aaa|aa|a", "aaaa"))

cat|dog          : ['cat', 'dog']
a|aa|aaa -> aaaa : ['a', 'a', 'a', 'a']
aaa|aa|a -> aaaa : ['aaa', 'a']


**结果分析：**

- `cat|dog` 在 `"cat dog bird CAT"` 中匹配到 `cat` 和 `dog`；`CAT` 是大写，而正则默认区分大小写，因此不匹配。
- 正则引擎采用“最左优先 + 分支按书写顺序尝试”的策略：`a|aa|aaa` 中 `a` 写在最前，每次都在最左位置优先匹配一个 `a`，于是 `"aaaa"` 得到 4 个 `"a"`。
- 把顺序改为 `aaa|aa|a` 后，每次优先匹配最长的 `aaa`，剩下一个 `a`，得到 `["aaa", "a"]`。这说明**分支顺序会影响匹配结果**。

In [4]:
text3 = "ct cat caat caaat"
for pattern in [r"ca*t", r"ca+t", r"ca?t", r"ca{2}t", r"ca{1,2}t"]:
    print(f"{pattern:10s} ->", re.findall(pattern, text3))

ca*t       -> ['ct', 'cat', 'caat', 'caaat']
ca+t       -> ['cat', 'caat', 'caaat']
ca?t       -> ['ct', 'cat']
ca{2}t     -> ['caat']
ca{1,2}t   -> ['cat', 'caat']


**量词含义与结果：**

| 模式 | 含义 | 匹配结果 |
| --- | --- | --- |
| `ca*t` | `a` 出现 0 次或多次 | `['ct', 'cat', 'caat', 'caaat']` |
| `ca+t` | `a` 出现 1 次或多次（至少 1 次） | `['cat', 'caat', 'caaat']` |
| `ca?t` | `a` 出现 0 次或 1 次 | `['ct', 'cat']` |
| `ca{2}t` | `a` 恰好出现 2 次 | `['caat']` |
| `ca{1,2}t` | `a` 出现 1 到 2 次 | `['cat', 'caat']` |

注意 `ca?t` 和 `ca{1,2}t` 不会匹配 `caaat`：因为 `a` 的数量超出上限后，模式无法在 `t` 前完成匹配；`ca{1,2}t` 也不能从 `caaat` 内部截取出匹配，因为必须以 `c` 开头。

## 实验2：分组、回溯引用与替换

In [5]:
text = "联系：alice@example.com；备选：bob@test.org"
emails = re.findall(r"([A-Za-z0-9._%+-]+)@([A-Za-z0-9.-]+\.[A-Za-z]{2,})", text)

for user, domain in emails:
    print(f"用户名: {user:10s} 域名: {domain}")

用户名: alice      域名: example.com
用户名: bob        域名: test.org


**规则说明：** 第 1 组 `([A-Za-z0-9._%+-]+)` 匹配 `@` 前的用户名；第 2 组 `([A-Za-z0-9.-]+\.[A-Za-z]{2,})` 匹配域名（含至少一个点，顶级域至少 2 个字母）。`re.findall` 在模式含有多个分组时返回元组列表。

In [6]:
text = "cat cat, dog cat, bird bird"
doubles = re.findall(r"\b(\w+)\s+\1\b", text)
print("连续重复的单词:", doubles)

for m in re.finditer(r"\b(\w+)\s+\1\b", text):
    print(f"匹配文本: {m.group()!r}  位置: {m.span()}  重复词: {m.group(1)}")

连续重复的单词: ['cat', 'bird']
匹配文本: 'cat cat'  位置: (0, 7)  重复词: cat
匹配文本: 'bird bird'  位置: (18, 27)  重复词: bird


**规则说明：** `(\w+)` 捕获一个单词，`\s+` 匹配中间的空白，`\1` 是**后向引用**，要求再次出现与第 1 组完全相同的内容，`\b` 保证单词边界。因此只能匹配 `cat cat` 和 `bird bird`，而 `dog cat` 虽然相邻但不是同一个词，不会匹配。

In [7]:
text = "2024-05-01\n2024/05/02\n2024.05.03"
converted = re.sub(r"(\d{4})[-/.](\d{2})[-/.](\d{2})", r"\2-\3-\1", text)

print("转换前:")
print(text)
print("\n转换后:")
print(converted)
print("\n行数是否一致:", len(text.splitlines()) == len(converted.splitlines()))

转换前:
2024-05-01
2024/05/02
2024.05.03

转换后:
05-01-2024
05-02-2024
05-03-2024

行数是否一致: True


**规则说明：** `(\d{4})` 捕获年、`(\d{2})` 捕获月、`(\d{2})` 捕获日，分隔符用字符集 `[-/.]` 同时兼容 `-`、`/`、`.` 三种写法。替换表达式 `\2-\3-\1` 按“月-日-年”重排；`re.sub` 只替换匹配到的日期部分，换行符不在匹配范围内，因此行数保持不变。

In [8]:
text = "联系我：123-456-7890 或 987.654.3210"
unified = re.sub(r"(\d{3})[.-](\d{3})[.-](\d{4})", r"(\1) \2-\3", text)
print(unified)

联系我：(123) 456-7890 或 (987) 654-3210


**规则说明：** 三个分组分别捕获区号、中间三位和末四位，分隔符 `[.-]` 同时兼容点和短横线；替换为 `(\1) \2-\3`，即统一成 `(123) 456-7890` 的格式。

## 实验3：使用 BeautifulSoup 解析 HTML

使用题目给定的 HTML 字符串，不依赖网页下载。

In [9]:
from bs4 import BeautifulSoup

html_doc = """
<html><head><title>实验课程</title></head><body>
<h1>数据分析</h1>
<p class="course">第六讲：正则表达式</p>
<p class="course">第七讲：结构化数据</p>
<a href="https://example.org/lecture06">讲义六</a>
<a href="https://example.org/lecture07">讲义七</a>
</body></html>
"""

soup = BeautifulSoup(html_doc, "html.parser")

print("页面标题文本:", soup.title.get_text())
h1 = soup.h1
print("h1 标签名:", h1.name)
print("h1 文本:", h1.get_text())

页面标题文本: 实验课程
h1 标签名: h1
h1 文本: 数据分析


In [10]:
courses = soup.find_all("p", class_="course")
links = soup.find_all("a")

print("class='course' 的段落数:", len(courses))
for p in courses:
    print("  段落文本:", p.get_text())

print("\n链接数:", len(links))
for a in links:
    print(f"  链接文字: {a.get_text()}  |  href: {a.get('href')}")

class='course' 的段落数: 2
  段落文本: 第六讲：正则表达式
  段落文本: 第七讲：结构化数据

链接数: 2
  链接文字: 讲义六  |  href: https://example.org/lecture06
  链接文字: 讲义七  |  href: https://example.org/lecture07


**解析结果说明：** `find_all("p", class_="course")` 通过标签名和 class 属性定位课程段落；`find_all("a")` 取出全部链接，`a.get_text()` 得到可见链接文字，`a.get("href")` 读取 href 属性。

In [11]:
print("整个文档 get_text() :", repr(soup.get_text()))
print("单个段落 get_text() :", repr(courses[0].get_text()))
print("单个链接 get_text() :", repr(links[0].get_text()))

print("\n去掉多余空白后:")
print("整个文档:", repr(soup.get_text(strip=True)))
print("单个段落:", repr(courses[0].get_text(strip=True)))
print("单个链接:", repr(links[0].get_text(strip=True)))

整个文档 get_text() : '\n实验课程\n数据分析\n第六讲：正则表达式\n第七讲：结构化数据\n讲义六\n讲义七\n\n'
单个段落 get_text() : '第六讲：正则表达式'
单个链接 get_text() : '讲义六'

去掉多余空白后:
整个文档: '实验课程数据分析第六讲：正则表达式第七讲：结构化数据讲义六讲义七'
单个段落: '第六讲：正则表达式'
单个链接: '讲义六'


**比较：** `soup.get_text()` 会递归拼接整个文档所有可见文本，并保留 HTML 源码中的换行与缩进空白；单个段落和单个链接的 `get_text()` 只返回该标签内部的文字，因此结果短得多。使用 `strip=True` 可以去掉每段文本首尾的空白，得到更干净的字符串（但相邻块之间的文本仍会直接相连）。